---
title: "Sprint 2: Algorytmy ANN w Niskich Wymiarach"
subtitle: "Ewaluacja metod wyszukiwania wektorowego dla problemu Particle Tracking"
author: "Zespół HEP Tracking"
date: today
format: 
  pdf:
    toc: true
    number-sections: true
    colorlinks: true
    fig-format: pdf
    fig-width: 8
    fig-height: 5
execute:
  echo: false
  warning: false
---

In [ ]:
%config InlineBackend.figure_format = 'pdf'
%load_ext autoreload
%autoreload 2

import sys
import gc
from pathlib import Path
import numpy as np
import faiss

notebook_dir = Path().resolve()
project_root = notebook_dir.parent
src_path = str(project_root / "src")

if src_path not in sys.path:
    sys.path.append(src_path)

from hep_tracking.models import knn_scipy_ckdtree
from hep_tracking.utils import pad_features, evaluate_ann_model, measure_execution_time
from hep_tracking.plots import plot_pareto_frontier, plot_crossover, plot_ann_scaling, plot_exact_vs_ann
from hep_tracking.ann_models import FaissIVFFlat, FaissIVFPQ, HnswGraph

print("Zależności załadowane pomyślnie!")

In [ ]:
data_dir = project_root / "data"
dataset_name = "dataset_hard_100k.npz"
candidates_name = "candidates_hard_100k.npz"

data_path = data_dir / dataset_name
candidates_path = data_dir / candidates_name

if not data_path.exists() or not candidates_path.exists():
    raise FileNotFoundError("Brak danych! Upewnij się, że wywołałeś `make candidates`.")

features = np.load(data_path)["X"]
n_queries = features.shape[0]

true_indices = np.load(candidates_path)["indices"]
k_neighbors = true_indices.shape[1]

print(f"Załadowano dane: {n_queries} punktów w przestrzeni {features.shape[1]}D")
print(f"Szukamy {k_neighbors} najbliższych sąsiadów.")

# Wprowadzenie
Celem drugiego sprintu projektu jest ocena wydajności algorytmów przybliżonego wyszukiwania najbliższych sąsiadów (ANN) w kontekście rekonstrukcji torów cząstek. Ze względu na ograniczenia sprzętowe akceleratorów (GPU) narzucające długość wektora $m \in \{1, 2, 3, 4, 8, 12...\}$, 5-wymiarowe dane wejściowe zostały rozszerzone do 8 wymiarów przy pomocy bezstratnej techniki Zero-Padding.

# Poszukiwanie Granicy Pareto (Recall vs QPS)
W pierwszym eksperymencie przeprowadzono przeszukiwanie siatki hiperparametrów (Grid Search) dla trzech głównych algorytmów: **FAISS IVFFlat**, **FAISS IVFPQ** (z Product Quantization) oraz **HNSW**. 

Celem było wyznaczenie kompromisu między czułością modelu (Recall) a jego przepustowością (Queries Per Second).

In [ ]:
print("=== Przygotowanie danych (Zero-Padding do 8D) ===")
features_5d = np.load(data_path)["X"]
features = pad_features(features_5d, target_dim=8)
print(f"Dane gotowe: {features.shape}")

USE_GPU = True

results = {
    "IVFFlat": {"recall": [], "qps": [], "labels": []},
    "IVFPQ": {"recall": [], "qps": [], "labels": []},
    "HNSW": {"recall": [], "qps": [], "labels": []}
}

nlist = 100

print("\n=== START IVFFlat ===")
for nprobe in [1, 2, 5, 10, 20, 50]:
    print(f" -> Inicjalizacja IVFFlat (nprobe={nprobe})...")
    model = FaissIVFFlat(nlist=nlist, nprobe=nprobe, use_gpu=USE_GPU)
    qps, recall = evaluate_ann_model(f"IVFFlat (nprobe={nprobe})", model, features, true_indices, k_neighbors)
    results["IVFFlat"]["qps"].append(qps)
    results["IVFFlat"]["recall"].append(recall)
    results["IVFFlat"]["labels"].append(f"np={nprobe}")
    del model
    gc.collect()

print("\n=== START IVFPQ (Teraz na GPU z m=8!) ===")
for nprobe in [1, 2, 5, 10, 20, 50, 100]:
    print(f" -> Inicjalizacja IVFPQ (nprobe={nprobe})...")
    model = FaissIVFPQ(nlist=nlist, m=8, nbits=8, nprobe=nprobe, use_gpu=USE_GPU)
    qps, recall = evaluate_ann_model(f"IVFPQ (nprobe={nprobe})", model, features, true_indices, k_neighbors)
    results["IVFPQ"]["qps"].append(qps)
    results["IVFPQ"]["recall"].append(recall)
    results["IVFPQ"]["labels"].append(f"np={nprobe}")
    del model
    gc.collect()

print("\n=== START HNSW ===")
m_graph = 16
for ef in [10, 20, 50, 100, 200]:
    print(f" -> Inicjalizacja HNSW (ef={ef})...")
    model = HnswGraph(m=m_graph, ef_construction=200, ef=ef)
    qps, recall = evaluate_ann_model(f"HNSW (ef={ef})", model, features, true_indices, k_neighbors)
    results["HNSW"]["qps"].append(qps)
    results["HNSW"]["recall"].append(recall)
    results["HNSW"]["labels"].append(f"ef={ef}")
    del model
    gc.collect()

print("\nCAŁY BENCHMARK ZOSTAŁ POMYŚLNIE ZAKOŃCZONY!")

In [ ]:
plot_pareto_frontier(
    results, 
    title="Wydajność algorytmów ANN: Recall vs. QPS (Przestrzeń 8D z Zero-Paddingiem)"
)

# Analiza Wąskiego Gardła: CPU vs GPU (Crossover N)
Karty graficzne oferują potężną moc zrównoleglonych obliczeń, jednak posiadają istotny narzut czasowy związany z transferem danych przez magistralę PCIe oraz kompilacją kerneli JIT. 

Poniższy eksperyment wyznacza punkt przecięcia (Crossover Point), w którym rozmiar zbioru danych $N$ staje się na tyle duży, że użycie GPU staje się uzasadnione czasowo.


In [ ]:
target_sizes = {"1k": 1000, "10k": 10000, "100k": 100000, "1M": 1000000}
mode = "hard"
k_neighbors = 5
nprobe = 5

cpu_times = []
gpu_times = []
valid_sizes = []

print("=== WYSZUKIWANIE PUNKTU PRZECIĘCIA (CROSSOVER N): CPU vs GPU ===")

for size_label, n_points in target_sizes.items():
    filename = data_dir / f"dataset_{mode}_{size_label}.npz"

    if not filename.exists():
        print(f"[POMINIĘTO] Plik {filename.name} nie istnieje.")
        continue

    print(f"\nWczytywanie zbioru danych {size_label} ({n_points} punktów)...")
    features_5d = np.load(filename)["X"]
    features = pad_features(features_5d, target_dim=8)

    nlist = min(100, int(np.sqrt(features.shape[0])))

    model_cpu = FaissIVFFlat(nlist=nlist, nprobe=nprobe, use_gpu=False)
    model_cpu.build(features)
    
    best_cpu_time = measure_execution_time(lambda: model_cpu.query(features, k_neighbors), num_runs=3, warmup_runs=1)
    cpu_times.append(best_cpu_time)
    del model_cpu
    gc.collect()

    model_gpu = FaissIVFFlat(nlist=nlist, nprobe=nprobe, use_gpu=True)
    model_gpu.build(features)
    
    best_gpu_time = measure_execution_time(lambda: model_gpu.query(features, k_neighbors), num_runs=3, warmup_runs=1)
    gpu_times.append(best_gpu_time)
    del model_gpu
    gc.collect()

    valid_sizes.append(n_points)
    print(f" -> CPU Time: {best_cpu_time*1000:.2f} ms")
    print(f" -> GPU Time: {best_gpu_time*1000:.2f} ms")

plot_crossover(valid_sizes, cpu_times, gpu_times, title="Crossover N: CPU vs GPU")

In [ ]:
filename = data_dir / "dataset_hard_1k.npz"
features_5d = np.load(filename)["X"]
features_full = pad_features(features_5d, target_dim=8)

micro_sizes = [10, 100, 200, 500, 1000]
cpu_micro = []
gpu_micro = []

print("=== SPRAWDZENIE NIŻSZYCH WARTOŚCI N (< 1000) ===")

for size in micro_sizes:
    features = features_full[:size]
    nlist = min(100, int(np.sqrt(features.shape[0])))

    model_cpu = FaissIVFFlat(nlist=nlist, nprobe=5, use_gpu=False)
    model_cpu.build(features)
    best_cpu = measure_execution_time(lambda: model_cpu.query(features, k_neighbors), num_runs=5, warmup_runs=1)
    cpu_micro.append(best_cpu)
    del model_cpu

    model_gpu = FaissIVFFlat(nlist=nlist, nprobe=5, use_gpu=True)
    model_gpu.build(features)
    best_gpu = measure_execution_time(lambda: model_gpu.query(features, k_neighbors), num_runs=5, warmup_runs=1)
    gpu_micro.append(best_gpu)
    del model_gpu

    print(f"N={size}: CPU = {best_cpu*1000:.3f} ms | GPU = {best_gpu*1000:.3f} ms")

plot_crossover(micro_sizes, cpu_micro, gpu_micro, title="Crossover N: CPU vs GPU")

# Skalowalność Metod Przybliżonych
Wykorzystując optymalne parametry odczytane z granicy Pareto (gwarantujące czułość na poziomie ok. 95%), przeprowadzono badanie skalowalności czasowej algorytmów przybliżonych w funkcji rosnącego rozmiaru zbioru danych ($10^3 \le N \le 10^6$).

In [ ]:
params = {
    "IVFFlat": {"nprobe": 20},
    "IVFPQ": {"nprobe": 20, "m": 8},
    "HNSW": {"ef": 20, "m_graph": 16}
}

run_algos = {
    "Exact_CPU": True,
    "Exact_GPU": True,
    "IVFFlat": True,
    "IVFPQ": True,
    "HNSW": True
}

results_time = {alg: [] for alg in run_algos.keys()}
valid_sizes = []

print(f"=== START EKSPERYMENTU: TIME vs N (USE_GPU dla FAISS ANN = {USE_GPU}) ===")

valid_datasets_count = sum(1 for size in target_sizes if (data_dir / f"dataset_{mode}_{size}.npz").exists())
active_algos_count = sum(run_algos.values())
total_tasks = valid_datasets_count * active_algos_count
current_task = 0

if total_tasks == 0:
    print("[BŁĄD] Brak zadań do wykonania (brak plików lub wszystkie algorytmy wyłączone).")
else:
    if run_algos["Exact_GPU"]:
        gpu_res = faiss.StandardGpuResources()

    for size_label, n_points in target_sizes.items():
        dataset_path = data_dir / f"dataset_{mode}_{size_label}.npz"
        if not dataset_path.exists():
            print(f"[POMIJAM] Nie znaleziono pliku {size_label}")
            continue
            
        print(f"\nŁadowanie zbioru danych {size_label}...")
        features_5d = np.load(dataset_path)["X"]
        
        features = pad_features(features_5d, target_dim=8)
        nlist = min(100, int(np.sqrt(features.shape[0])))
        valid_sizes.append(n_points)
        
        if run_algos["Exact_CPU"]:
            index_exact_cpu = faiss.IndexFlatL2(features.shape[1])
            index_exact_cpu.add(features)
            results_time["Exact_CPU"].append(measure_execution_time(lambda: index_exact_cpu.search(features, k_neighbors)))
            del index_exact_cpu
            current_task += 1
            print(f"[{current_task}/{total_tasks}] Exact_CPU zakończone")
        else:
            results_time["Exact_CPU"].append(np.nan)

        if run_algos["Exact_GPU"]:
            index_exact_gpu = faiss.IndexFlatL2(features.shape[1])
            index_exact_gpu = faiss.index_cpu_to_gpu(gpu_res, 0, index_exact_gpu)
            index_exact_gpu.add(features)
            results_time["Exact_GPU"].append(measure_execution_time(lambda: index_exact_gpu.search(features, k_neighbors)))
            del index_exact_gpu
            current_task += 1
            print(f"[{current_task}/{total_tasks}] Exact_GPU zakończone")
        else:
            results_time["Exact_GPU"].append(np.nan)

        if run_algos["IVFFlat"]:
            model_ivf = FaissIVFFlat(nlist=nlist, nprobe=params["IVFFlat"]["nprobe"], use_gpu=USE_GPU)
            model_ivf.build(features)
            results_time["IVFFlat"].append(measure_execution_time(lambda: model_ivf.query(features, k_neighbors)))
            del model_ivf
            current_task += 1
            print(f"[{current_task}/{total_tasks}] IVFFlat zakończone")
        else:
            results_time["IVFFlat"].append(0)
        
        if run_algos["IVFPQ"]:
            model_pq = FaissIVFPQ(nlist=nlist, m=params["IVFPQ"]["m"], nbits=8, nprobe=params["IVFPQ"]["nprobe"], use_gpu=USE_GPU)
            model_pq.build(features)
            results_time["IVFPQ"].append(measure_execution_time(lambda: model_pq.query(features, k_neighbors)))
            del model_pq
            current_task += 1
            print(f"[{current_task}/{total_tasks}] IVFPQ zakończone")
        else:
            results_time["IVFPQ"].append(0)

        if run_algos["HNSW"]:
            model_hnsw = HnswGraph(m=params["HNSW"]["m_graph"], ef_construction=200, ef=params["HNSW"]["ef"])
            model_hnsw.build(features)
            results_time["HNSW"].append(measure_execution_time(lambda: model_hnsw.query(features, k_neighbors)))
            del model_hnsw
            current_task += 1
            print(f"[{current_task}/{total_tasks}] HNSW zakończone")
        else:
            results_time["HNSW"].append(0)
        
        gc.collect()
        
    gpu_label = "GPU" if USE_GPU else "CPU"
    plot_ann_scaling(
        valid_sizes, 
        results_time, 
        use_gpu=USE_GPU,
        title="Przestrzeń 8D: Algorytmy Exact (100%) vs Przybliżone ANN"
    )

# Weryfikacja Założeń: Exact kNN vs ANN w Niskich Wymiarach
Ostatni etap analizy to weryfikacja celowości stosowania algorytmów heurystycznych (ANN) w przestrzeniach o bardzo małej liczbie wymiarów (5D). Do zestawienia dołączono precyzyjne metody `FAISS ExactL2` oraz strukturę podziału przestrzeni `Scipy cKDTree` (100% Recall).

In [ ]:
target_sizes = {"10k": 10000, "100k": 100000, "1M": 1000000}
mode = "hard"
k_neighbors = 5

run_algos = {
    "Exact_CPU": True,
    "Exact_GPU": True,
    "cKDTree_CPU": True,
    "IVFFlat_GPU": True,
    "HNSW_CPU": True
}

results_time = {alg: [] for alg in run_algos.keys()}
valid_sizes = []

print("=== START EKSPERYMENTU: EXACT kNN vs ANN (Przestrzeń 5D) ===")

valid_datasets_count = sum(1 for size in target_sizes if (data_dir / f"dataset_{mode}_{size}.npz").exists())
active_algos_count = sum(run_algos.values())
total_tasks = valid_datasets_count * active_algos_count
current_task = 0

if total_tasks == 0:
    print("[BŁĄD] Brak zadań do wykonania (brak plików lub wszystkie algorytmy wyłączone).")
else:
    if run_algos["Exact_GPU"] or run_algos["IVFFlat_GPU"]:
        gpu_res = faiss.StandardGpuResources()

    for size_label, n_points in target_sizes.items():
        dataset_path = data_dir / f"dataset_{mode}_{size_label}.npz"
        if not dataset_path.exists():
            print(f"[POMIJAM] Brak pliku: {size_label}")
            continue
            
        print(f"\nŁadowanie zbioru danych {size_label} (Surowe 5D)...")
        features = np.ascontiguousarray(np.load(dataset_path)["X"], dtype=np.float32)
        valid_sizes.append(n_points)
        
        nlist = min(100, int(np.sqrt(features.shape[0])))
        
        if run_algos["Exact_CPU"]:
            index_exact_cpu = faiss.IndexFlatL2(features.shape[1])
            index_exact_cpu.add(features)
            results_time["Exact_CPU"].append(measure_execution_time(lambda: index_exact_cpu.search(features, k_neighbors)))
            del index_exact_cpu
            current_task += 1
            print(f"[{current_task}/{total_tasks}] Exact_CPU zakończone")
        else:
            results_time["Exact_CPU"].append(0)

        if run_algos["Exact_GPU"]:
            index_exact_gpu = faiss.IndexFlatL2(features.shape[1])
            index_exact_gpu = faiss.index_cpu_to_gpu(gpu_res, 0, index_exact_gpu)
            index_exact_gpu.add(features)
            results_time["Exact_GPU"].append(measure_execution_time(lambda: index_exact_gpu.search(features, k_neighbors)))
            del index_exact_gpu
            current_task += 1
            print(f"[{current_task}/{total_tasks}] Exact_GPU zakończone")
        else:
            results_time["Exact_GPU"].append(0)
        
        if run_algos["cKDTree_CPU"]:
            results_time["cKDTree_CPU"].append(measure_execution_time(lambda: knn_scipy_ckdtree(features, k_neighbors)))
            current_task += 1
            print(f"[{current_task}/{total_tasks}] cKDTree_CPU zakończone")
        else:
            results_time["cKDTree_CPU"].append(0)

        if run_algos["IVFFlat_GPU"]:
            model_ivf = FaissIVFFlat(nlist=nlist, nprobe=20, use_gpu=True)
            model_ivf.build(features)
            results_time["IVFFlat_GPU"].append(measure_execution_time(lambda: model_ivf.query(features, k_neighbors)))
            del model_ivf
            current_task += 1
            print(f"[{current_task}/{total_tasks}] IVFFlat_GPU zakończone")
        else:
            results_time["IVFFlat_GPU"].append(0)

        if run_algos["HNSW_CPU"]:
            model_hnsw = HnswGraph(m=16, ef_construction=200, ef=20)
            model_hnsw.build(features)
            results_time["HNSW_CPU"].append(measure_execution_time(lambda: model_hnsw.query(features, k_neighbors)))
            del model_hnsw
            current_task += 1
            print(f"[{current_task}/{total_tasks}] HNSW_CPU zakończone")
        else:
            results_time["HNSW_CPU"].append(0)

        gc.collect()

    plot_exact_vs_ann(valid_sizes, results_time, title="Wymiar 5D: Czy opłaca się używać metod przybliżonych (ANN)?")

## Wnioski Końcowe (Curse of Dimensionality)

Wyniki powyższego starcia dostarczają kluczowych wniosków inżynieryjnych:

1. **Deklasacja przez metody oparte na drzewach:** W przestrzeni 5-wymiarowej matematycznie dokładny algorytm `cKDTree` (złożoność $O(N \log N)$) deklasuje zaawansowane metody przybliżone.
2. **Metody ANN:** Z metod przybliżonych ANN IVFFlat zdaje się radzić sobie najlepiej. Możliwe, że dla zbiorów danych o większej liczebności lub liczbie wymiarów uzyskałby wyniki lepsze od cKDTree 
